In [1]:
DELETE FROM gold_factFlight
WHERE DateKey BETWEEN 20240700 AND 20240799;

StatementMeta(, 044b673c-3b65-4373-8882-9c06bc8ef95e, 2, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

# **0) Config & setup**

In [1]:

# Fabric / Spark notebook: Bronze→Silver→Gold transformations
# This version fixes row-loss in Fact by:
# - avoiding inner joins that drop cancelled/diverted flights
# - not filtering out rows with null arrival/departure times unnecessarily
# - idempotent MERGE to Fact by a stable LineageID (hash of business grain)
# - consistent typing and trimming to match SQL style

# ---------- CONFIG ----------
BRONZE = "Bronze_Flights"
SILVER = "Silver_Flights"

G_DIMDATE = "Gold_DimDate"
G_DIMCARR = "Gold_DimCarrier"
G_DIMAIR  = "Gold_DimAirport"
G_FACT    = "Gold_FactFlight"

# Optional load window (inclusive). Set to None to process all.
LOAD_START = "2024-07-01"  # e.g., "2024-07-01"
LOAD_END   = "2024-07-31"  # e.g., "2024-07-31"

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException

def table_exists(name: str) -> bool:
    try:
        spark.table(name)
        return True
    except AnalysisException:
        return False

# Ensure Fact has LineageID column for idempotent upserts
if table_exists(G_FACT):
    try:
        spark.table(G_FACT).select("LineageID").limit(1).collect()
        print(f"{G_FACT}.LineageID present.")
    except AnalysisException:
        spark.sql(f"ALTER TABLE {G_FACT} ADD COLUMNS (LineageID string)")
        print(f"Added LineageID to {G_FACT}.")


StatementMeta(, 6b0a2ade-f21c-4de3-a5cb-a07fb6bec085, 3, Finished, Available, Finished)

Gold_FactFlight.LineageID present.


# **1) Bronze ➜ Silver (normalization, typing, minimal rules)**

In [2]:

b = spark.table(BRONZE)

# Some BTS exports use OP_UNIQUE_CARRIER, some CARRIER
carrier_col = "OP_UNIQUE_CARRIER" if "OP_UNIQUE_CARRIER" in b.columns else "CARRIER"

# Normalize times from HHMM (ints like 945, 1630) to string "HH:mm:ss"
def hhmm_to_timestr(col):
    h = (col / F.lit(100)).cast("int")
    m = (col % F.lit(100)).cast("int")
    ok = (h >= 0) & (h <= 23) & (m >= 0) & (m <= 59)
    return F.when(col.isNull() | (~ok), F.lit(None)) \
            .otherwise(F.format_string("%02d:%02d:00", h, m))

# Coalesce column names that sometimes vary by dump
def col_or_null(df, name):
    return F.col(name) if name in df.columns else F.lit(None).alias(name)

df = (b
    # Base typed/trimmed fields
    .withColumn("FlightDate", F.to_date(F.col("FL_DATE")))
    .withColumn("CarrierCode", F.upper(F.trim(F.col(carrier_col))))
    .withColumn("OriginAirportID", F.col("ORIGIN_AIRPORT_ID").cast("int"))
    .withColumn("OriginCode", F.upper(F.trim(F.col("ORIGIN"))))
    .withColumn("OriginCity", F.trim(F.col("ORIGIN_CITY_NAME")))
    .withColumn("DestAirportID", F.col("DEST_AIRPORT_ID").cast("int"))
    .withColumn("DestCode", F.upper(F.trim(F.col("DEST"))))
    .withColumn("DestCity", F.trim(F.col("DEST_CITY_NAME")))
    .withColumn("DepTimeHHMM", F.col("DEP_TIME").cast("int"))
    .withColumn("ArrTimeHHMM", F.col("ARR_TIME").cast("int"))
    .withColumn("DepTime", hhmm_to_timestr(F.col("DEP_TIME")))
    .withColumn("ArrTime", hhmm_to_timestr(F.col("ARR_TIME")))
    .withColumn("DepDelayMin", F.col("DEP_DELAY").cast("int"))
    .withColumn("ArrDelayMin", F.col("ARR_DELAY").cast("int"))
    .withColumn("Cancelled", F.coalesce(F.col("CANCELLED").cast("int"), F.lit(0)))
    .withColumn("CancellationCode", F.upper(F.trim(col_or_null(b, "CANCELLATION_CODE"))))
    .withColumn("Diverted", F.coalesce(F.col("DIVERTED").cast("int"), F.lit(0)))
    .withColumn("DistanceMiles", F.col("DISTANCE").cast("int"))
)

# Optional: filter to a window at Silver stage (keeps demo fast)
if LOAD_START and LOAD_END:
    df = df.filter((F.col("FlightDate") >= F.lit(LOAD_START)) & (F.col("FlightDate") <= F.lit(LOAD_END)))

# Remove obvious junk rows ONLY if FlightDate or CarrierCode is missing
# (Do NOT drop rows just because times or delays are null — that caused missing facts)
df = df.filter(F.col("FlightDate").isNotNull() & F.col("CarrierCode").isNotNull())

# Build a stable lineage key to make fact upserts idempotent.
# Use business grain components; include raw HHMM and airports to avoid collisions.
lineage_expr = F.concat_ws(
    "||",
    F.date_format("FlightDate","yyyy-MM-dd"),
    F.coalesce(F.col("CarrierCode"), F.lit("")),
    F.coalesce(F.col("OriginCode"), F.lit("")),
    F.coalesce(F.col("DestCode"), F.lit("")),
    F.coalesce(F.col("DepTimeHHMM").cast("string"), F.lit("")),
    F.coalesce(F.col("ArrTimeHHMM").cast("string"), F.lit("")),
    F.coalesce(F.col("DistanceMiles").cast("string"), F.lit("")),
    F.coalesce(F.col("Cancelled").cast("string"), F.lit("")),
    F.coalesce(F.col("Diverted").cast("string"), F.lit(""))
)
df = df.withColumn("LineageID", F.sha2(lineage_expr, 256))

silver = df.select(
    "FlightDate","CarrierCode",
    "OriginAirportID","OriginCode","OriginCity",
    "DestAirportID","DestCode","DestCity",
    "DepTimeHHMM","DepTime","ArrTimeHHMM","ArrTime",
    "DepDelayMin","ArrDelayMin",
    "Cancelled","CancellationCode","Diverted","DistanceMiles",
    "LineageID"
)

silver.write.mode("append").format("delta").saveAsTable(SILVER)
print("Silver append complete. Rows now:", spark.table(SILVER).count())


StatementMeta(, 6b0a2ade-f21c-4de3-a5cb-a07fb6bec085, 4, Finished, Available, Finished)

AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'FlightDate' and 'FlightDate'

# **2) Silver ➜ Gold (Dims upsert, Fact MERGE idempotent)**

In [ ]:

sf_all = spark.table(SILVER)

if LOAD_START and LOAD_END:
    sf = sf_all.filter((F.col("FlightDate") >= F.lit(LOAD_START)) & (F.col("FlightDate") <= F.lit(LOAD_END)))
else:
    sf = sf_all

# ---- Dimensions ----
# DimDate
dates = (sf.select("FlightDate").where(F.col("FlightDate").isNotNull()).distinct()
    .withColumn("DateKey", F.date_format("FlightDate","yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("FlightDate"))
    .withColumn("Month", F.month("FlightDate"))
    .withColumn("Day", F.dayofmonth("FlightDate"))
    .withColumn("MonthName", F.date_format("FlightDate","MMMM"))
    .withColumn("Quarter", F.quarter("FlightDate"))
)
spark.sql(f"CREATE TABLE IF NOT EXISTS {G_DIMDATE} (DateKey int, FlightDate date, Year int, Month int, Day int, MonthName string, Quarter int) USING delta")
(dates
 .select("DateKey","FlightDate","Year","Month","Day","MonthName","Quarter")
 .write.mode("append").format("delta").saveAsTable(G_DIMDATE))

# DimCarrier
carriers = (sf.select("CarrierCode").where(F.col("CarrierCode").isNotNull()).distinct()
    .withColumn("CarrierName", F.lit(None).cast("string"))  # placeholder if no lookup available
)
spark.sql(f"CREATE TABLE IF NOT EXISTS {G_DIMCARR} (CarrierCode string, CarrierName string) USING delta")
carriers.write.mode("append").format("delta").saveAsTable(G_DIMCARR)

# DimAirport
airports = (
    sf.select(
        F.col("OriginAirportID").alias("AirportID"),
        F.col("OriginCode").alias("AirportCode"),
        F.col("OriginCity").alias("City")
    ).where(F.col("AirportCode").isNotNull()).distinct()
    .unionByName(
        sf.select(
            F.col("DestAirportID").alias("AirportID"),
            F.col("DestCode").alias("AirportCode"),
            F.col("DestCity").alias("City")
        ).where(F.col("AirportCode").isNotNull()).distinct()
    ).dropDuplicates(["AirportCode"])
)
spark.sql(f"CREATE TABLE IF NOT EXISTS {G_DIMAIR} (AirportID int, AirportCode string, City string) USING delta")
airports.write.mode("append").format("delta").saveAsTable(G_DIMAIR)

# ---- Fact ----
# Build surrogate keys for date (int) and keep codes for carrier/airports (star light)
fact = (sf
    .withColumn("DateKey", F.date_format("FlightDate","yyyyMMdd").cast("int"))
    .select(
        "LineageID","DateKey",
        "CarrierCode",
        F.col("OriginCode").alias("OriginCode"),
        F.col("DestCode").alias("DestCode"),
        "DepTime","ArrTime",
        "DepDelayMin","ArrDelayMin",
        "Cancelled","Diverted",
        "DistanceMiles"
    )
)

# Create Fact table if missing
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G_FACT} (
    LineageID string,
    DateKey int,
    CarrierCode string,
    OriginCode string,
    DestCode string,
    DepTime string,
    ArrTime string,
    DepDelayMin int,
    ArrDelayMin int,
    Cancelled int,
    Diverted int,
    DistanceMiles int
) USING delta
""")

# Idempotent upsert by LineageID
fact.createOrReplaceTempView("src_fact")

spark.sql(f"""
MERGE INTO {G_FACT} AS T
USING src_fact AS S
ON T.LineageID = S.LineageID
WHEN MATCHED THEN UPDATE SET
  T.DateKey      = S.DateKey,
  T.CarrierCode  = S.CarrierCode,
  T.OriginCode   = S.OriginCode,
  T.DestCode     = S.DestCode,
  T.DepTime      = S.DepTime,
  T.ArrTime      = S.ArrTime,
  T.DepDelayMin  = S.DepDelayMin,
  T.ArrDelayMin  = S.ArrDelayMin,
  T.Cancelled    = S.Cancelled,
  T.Diverted     = S.Diverted,
  T.DistanceMiles= S.DistanceMiles
WHEN NOT MATCHED THEN INSERT *
""")

print("Gold upsert complete.")


StatementMeta(, 6b0a2ade-f21c-4de3-a5cb-a07fb6bec085, -1, Cancelled, , Cancelled)

# **3) Quick checks**

In [3]:

def cnt(t): 
    try:
        return spark.table(t).count()
    except Exception:
        return 0

print("Counts:",
      "Silver", cnt(SILVER),
      "DimDate", cnt(G_DIMDATE),
      "DimCarrier", cnt(G_DIMCARR),
      "DimAirport", cnt(G_DIMAIR),
      "Fact", cnt(G_FACT))

# Sanity: include cancelled/diverted without arrival times
cancel_no_arr = spark.table(G_FACT).filter(
    (F.col("Cancelled")==1) & (F.col("ArrTime").isNull())
).count()
print("Cancelled rows w/o ArrTime (not an error):", cancel_no_arr)

# Window check if defined
if LOAD_START and LOAD_END:
    min_key = int(LOAD_START.replace("-",""))
    max_key = int(LOAD_END.replace("-",""))
    in_window = spark.table(G_FACT).filter((F.col("DateKey")>=min_key)&(F.col("DateKey")<=max_key)).count()
    print(f"Fact rows in window [{min_key}..{max_key}]:", in_window)


StatementMeta(, 6b0a2ade-f21c-4de3-a5cb-a07fb6bec085, 5, Finished, Available, Finished)

Counts: Silver 634613 DimDate 213 DimCarrier 15 DimAirport 344 Fact 4721017


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ArrTime` cannot be resolved. Did you mean one of the following? [`DateKey`, `DestKey`, `OriginKey`, `ArrDelayMin`, `CarrierKey`].;
'Filter (Cancelled#3868 AND isnull('ArrTime))
+- SubqueryAlias spark_catalog.flights.Gold_FactFlight
   +- Relation spark_catalog.flights.gold_factflight[FactFlightId#3861L,DateKey#3862,CarrierKey#3863,OriginKey#3864,DestKey#3865,DepDelayMin#3866,ArrDelayMin#3867,Cancelled#3868,Diverted#3869,DistanceMiles#3870,StagingFlightID#3871L,LineageID#3872] parquet
